# UC-MAPS-1 — Render a Map Image via OGC API - Maps

**Persona:** GIS Analyst

**Goal:** Ingest a set of administrative polygon features via OGC Features,
apply a preset for the maps extension, then call `GET /{dataset}/map` to
retrieve a rendered PNG image of the catalog, and finish with a deep link
to the map viewer web page.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `maps` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC API - Maps 1.0 (OGC 20-058)
- Routes: `/maps/`, `/maps/{dataset}`, `/maps/{dataset}/map`

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-maps")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "admin-regions")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")

## Step 1 — Check maps landing page

`GET /maps/` returns the OGC API - Maps landing page with links to available
datasets. A 200 response confirms the extension is active.

In [ ]:
r = client.get("/maps/")
print(f"status: {r.status_code}")

_maps_active = r.status_code == 200

if _maps_active:
    landing = r.json()
    print(f"title  : {landing.get('title', 'n/a')}")
    links = landing.get("links", [])
    print(f"links  : {len(links)}")
    for lnk in links[:5]:
        print(f"  rel={lnk.get('rel')}  href={lnk.get('href')}")
elif r.status_code == 404:
    print("SKIP: /maps/ endpoint not found — maps extension may not be active.")
else:
    print(r.text[:300])

## Step 2 — Create catalog and collection, ingest features

The maps extension renders features that already exist in the platform. We
create a demo catalog and collection via OGC Features and POST a few polygon
features covering central Italy.

In [ ]:
# --- catalog ---
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "Maps demo catalog"}),
)
print(f"catalog   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- collection ---
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({
        "id": COLLECTION_ID,
        "title": "Administrative regions demo",
    }),
)
print(f"collection: {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- items ---
regions = [
    {"id": "lazio",   "name": "Lazio",   "coords": [[11.5,41.2],[13.9,41.2],[13.9,42.9],[11.5,42.9],[11.5,41.2]]},
    {"id": "toscana", "name": "Toscana", "coords": [[9.7,42.4],[12.4,42.4],[12.4,44.5],[9.7,44.5],[9.7,42.4]]},
    {"id": "umbria",  "name": "Umbria",  "coords": [[11.9,42.3],[13.3,42.3],[13.3,43.7],[11.9,43.7],[11.9,42.3]]},
]

for reg in regions:
    feat = {
        "type": "Feature",
        "id": reg["id"],
        "geometry": {"type": "Polygon", "coordinates": [reg["coords"]]},
        "properties": {"name": reg["name"], "country": "IT"},
    }
    r = client.post(
        f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
        content=json.dumps(feat),
    )
    if r.status_code in (200, 201):
        print(f"  item '{reg['id']}': created")
    elif r.status_code == 409:
        print(f"  item '{reg['id']}': already exists")
    else:
        print(f"  item '{reg['id']}': {r.status_code} {r.text[:200]}")

## Step 3 — List datasets available for maps

`GET /maps/{dataset}` returns the dataset descriptor, including links to the
map endpoint. A 200 confirms the catalog is visible to the maps extension.

In [ ]:
if not _maps_active:
    print("SKIP: maps extension not active.")
else:
    r = client.get(f"/maps/{CATALOG_ID}")
    print(f"dataset descriptor status: {r.status_code}")
    if r.status_code == 200:
        ds = r.json()
        print(f"  title: {ds.get('title', 'n/a')}")
        for lnk in ds.get("links", []):
            print(f"  link: rel={lnk.get('rel')}  href={lnk.get('href')}")
    elif r.status_code == 404:
        print(f"  404: catalog '{CATALOG_ID}' not found — run step 2 first.")
    else:
        print(r.text[:300])

## Step 4 — Render map as PNG

`GET /maps/{dataset}/map?collections={col}&bbox={bbox}&crs=EPSG:3857&f=png`
returns a rendered PNG image of the features in the specified bounding box.
The `collections` query parameter selects which collections to include;
`bbox` is the geographic extent in CRS84 (lon,lat).

In [ ]:
if not _maps_active:
    print("SKIP: maps extension not active.")
else:
    img_headers = {"Authorization": f"Bearer {TOKEN}"}
    img_client  = httpx.Client(base_url=BASE_URL, headers=img_headers, timeout=60.0)

    r = img_client.get(
        f"/maps/{CATALOG_ID}/map",
        params={
            "collections": COLLECTION_ID,
            "bbox": "9.0,40.0,15.0,46.0",
            "crs": "EPSG:3857",
            "width": 512,
            "height": 512,
            "f": "png",
        },
    )
    print(f"map render status: {r.status_code}")

    if r.status_code == 200:
        ct = r.headers.get("content-type", "")
        print(f"content-type : {ct}")
        print(f"body size    : {len(r.content)} bytes")
        assert "image" in ct or "octet-stream" in ct, f"Unexpected content-type: {ct}"
        print("Map PNG received.")
    elif r.status_code == 404:
        print("  404 — catalog or collection not found.")
    elif r.status_code == 422:
        print("  422 — validation error (check bbox/crs parameters).")
        print(r.text[:300])
    else:
        print(r.text[:300])
    img_client.close()

## Step 5 — Maps conformance

`GET /maps/conformance` lists the OGC API - Maps conformance classes declared
by this deployment.

In [ ]:
if not _maps_active:
    print("SKIP: maps extension not active.")
else:
    r = client.get("/maps/conformance")
    print(f"conformance status: {r.status_code}")
    if r.status_code == 200:
        uris = r.json().get("conformsTo", [])
        print(f"Conformance classes ({len(uris)}):")
        for uri in uris:
            print(f"  {uri}")
    else:
        print(r.text[:300])

## Teardown

Remove the demo collection and catalog created in step 2.

In [ ]:
_TEARDOWN = os.environ.get("MAPS_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection: {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set MAPS_TEARDOWN=true to enable).")

## Result — Open the map viewer

In [ ]:
client.close()
print(f"Open the map viewer: {BASE_URL}/web/pages/map_viewer?language=en")
print(f"Direct map URL:")
print(f"  {BASE_URL}/maps/{CATALOG_ID}/map?collections={COLLECTION_ID}&bbox=9.0,40.0,15.0,46.0&crs=EPSG:3857&f=png")